In [1]:
import pandas as pd
from tqdm import tqdm
import time

In [2]:
df_true = pd.read_csv('./splits/midwest-big-train.csv')

In [35]:
def proportional_sample(df, col_name, sample_size):
    value_counts = df[col_name].value_counts(normalize=True)
    sampled_counts = (value_counts * sample_size).round().astype(int)

    sampled_dfs = []
    for value, count in sampled_counts.items():
        subset = df[df[col_name] == value].sample(n=min(count, len(df[df[col_name] == value])), replace=False) #sample without replacement, take the minimum of the count or the amount of that value in the df.
        sampled_dfs.append(subset)

    sampled_df = pd.concat(sampled_dfs)
    return sampled_df

## Take a proportional slice of the text to practice with

sample_size = 1000000
df = proportional_sample(df_true, 'rating', sample_size)

In [36]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def pnn(n):
    if n in {1,2}: return 0
    elif n in {3,4}: return 1
    else: return 2


def preprocess_text(text):
    # Maybe there's a better way to do this, but it gets hung up on reviews that are just integers 
    # I don't know why someone would leave such a review but there are a few
    text = str(text) 
    # Make lowercase and remove puncuation 
    text = text.lower()
    text = text.replace('translated by google', '')
    text = text.replace('\n', ' ')
    text = re.sub(r'[^\w\s]', '', text)
    
    return text

def minus1(n):
    return n-1

# Turn ratings into +/-/neutral and do a bit of light processing to the text
df['pnn'] = df['rating'].apply(pnn)
df['text'] = df['text'].apply(preprocess_text)
df['rating'] = df['rating'].apply(minus1)

In [37]:
df.sample(n=20)

,rating,text,type,pnn
8833933,0,the cashier had a terrible cough hacking all o...,Fast food restaurant,0
4463180,4,the culvers root beer is my favorite,Restaurant,2
106635,4,my boys favorite place could eat here 5 times ...,Takeout Restaurant,2
6951990,3,mexican restaurant very good food and varied ...,Restaurant,1
5547952,4,the food was great and the service was as well...,Vegetarian restaurant,2
1841816,4,great pizza and wings,Breakfast restaurant,2
8132028,2,steve the window guy was enthusiastic to serve...,Restaurant,1
8149152,2,slow never get a receipt so you cant do their ...,Restaurant,1
8002157,2,i love hamburgers with their chips and also t...,Restaurant,1
1688162,4,the price is right and the food is righteouser,Puerto Rican restaurant,2


In [20]:
from transformers import BertTokenizer, BertModel
from transformers import BertForSequenceClassification
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Base bert model should be sufficient. 
model_name = "bert-base-uncased" 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

#print(model)
print('Device: ', device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device:  cuda


This chunk is for preliminary testing, it can be ignored

In [14]:
def tokenize_text(text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    return inputs

In [15]:
def predict_sentiment(text):
    inputs = tokenize_text(text)
    inputs = {key: val.to(device) for key, val in inputs.items()} #move the inputs to the correct device.
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
    return predicted_class

In [18]:
# Simple test cycle of the untrained model
review = 'The best chicken ever'
label = predict_sentiment(review)
meaning = ['negative', 'neutral/average', 'good']

print('Review: ', review)
print(f'Predicted rating: {label} {meaning[label]}')

Review:  The best chicken ever
Predicted rating: 1 neutral/average


Run this so that the training autogenerates a plot after finishing


In [7]:
import datetime
import matplotlib.pyplot as plt
import numpy as np

def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    # Create the plot
    plt.figure(figsize=(10, 6))  # Adjust figure size if needed

    # Plot List 1
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')

    # Plot List 2
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')

    current_time = datetime.datetime.now()
    
    # Add Labels and Title
    plt.xlabel("Epoch") 
    plt.ylabel("Accuracy (%)")
    plt.title(f"Bert - Midwest data ({current_time.strftime("%Y-%m-%d %H:%m")})")

    # Add Legend
    plt.legend()

    # Add Grid (Optional)
    plt.grid(True)

    
    file_name = f'model-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    # Show the Plot
    plt.show()

Training the BERT model begins here

In [21]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=25): #add max_length.
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length #store max length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        inputs = self.tokenizer(
            text,
            padding="max_length", #important
            truncation=True, #important
            max_length=self.max_length, #important
            return_tensors="pt"
        )
        input_ids = inputs['input_ids'].flatten()
        attention_mask = inputs['attention_mask'].flatten()
        label_tensor = torch.tensor(label)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_tensor
        }

In [40]:
# Export the text and ratings to a list, this part is necessary otherwise pandas keeps the index
X = df['text'].values.tolist()
y = df['pnn'].values.tolist()

# The batch size seems to need to be pretty small, anything more than 32 crashed my 3070 with 8GB VRAM
# monitor RAM use in the terminal with "watch -n5 nvidia-smi"
batch_size = 32

# Remember to stratify based on the ratings
train_texts, test_texts, train_labels, test_labels = train_test_split(X, y, test_size=.2, stratify=y)

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

optimizer = torch.optim.AdamW(model.parameters(), lr=.001)

In [38]:
from transformers import Trainer, TrainingArguments

for param in model.bert.parameters():
    param.requires_grad = False
    
training_args = TrainingArguments(
    output_dir="./results",  # Directory to save model checkpoints and logs
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=32,  # Batch size for evaluation
    warmup_steps=500,  # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,  # Weight decay for regularization
    logging_dir="./logs",  # Directory for storing logs
    logging_steps=100,  # Log training information every 100 steps
    eval_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Save a model checkpoint at the end of each epoch
    load_best_model_at_end=True,  # Load the best model (based on eval_loss) at the end of training
    metric_for_best_model="eval_loss", # Use eval_loss to determine the best model.
    greater_is_better=False, #we want the lowest eval_loss
    learning_rate=2e-5,  # Learning rate
    save_total_limit=1, # Only save the best model.
)

In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    labels = p.label_ids
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='weighted', zero_division=0)
    recall = recall_score(labels, preds, average='weighted', zero_division=0)
    f1 = f1_score(labels, preds, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [41]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  # Pass the compute_metrics function
)

/tmp/ipykernel_467785/102837061.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()
results = trainer.evaluate()
print(results)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.541500,0.526544,0.767800,0.759825,0.767800,0.754439
2,0.507900,0.526722,0.767450,0.759035,0.767450,0.755604
3,0.495200,0.526497,0.767885,0.759687,0.767885,0.755258
